In [0]:
%pip install lightgbm

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install optuna

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import random
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_absolute_error

FORECAST_DATE   = pd.to_datetime('2010-08-14')               
BACKTEST_DATE   = FORECAST_DATE - pd.Timedelta(days=1)        
TOTAL_TRIALS    = 25                                           
ERRORS_PATH      = '/dbfs/mnt/thesis/mlops/lgbm/'

random.seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [0]:
def create_features(df):
    df = df.copy()
    df['hour']      = df['DATETIME'].dt.hour
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_of_week'] = df['DATETIME'].dt.dayofweek
    df['month']       = df['DATETIME'].dt.month
    for lag in (1, 12, 288, 2016):
        df[f'lag_{lag}'] = df.groupby('LOCATION')['VALUE'].shift(lag)
    grp = df.groupby('LOCATION')['VALUE']
    df['roll_mean_288'] = grp.transform(lambda x: x.shift(1).rolling(288).mean())
    df['roll_std_288']  = grp.transform(lambda x: x.shift(1).rolling(288).std())
    df.dropna(inplace=True)
    return df

#Prepare data

input_path = '/dbfs/mnt/thesis/output_data/'
df_raw = (
    pd.read_csv(input_path + "processed_data.csv", parse_dates=['DATETIME'])
      .sort_values(['LOCATION','DATETIME'])
      .reset_index(drop=True)
)
df = create_features(df_raw)
df['LOCATION'] = df['LOCATION'].astype('category')

FEATURE_COLS = [
    'hour_sin','hour_cos','day_of_week','month',
    'lag_1','lag_12','lag_288','lag_2016',
    'roll_mean_288','roll_std_288',
    'LOCATION'
]
CATEGORICAL_FEATURES = ['LOCATION','day_of_week','month']
TARGET_COL = 'VALUE'

val_start = BACKTEST_DATE - pd.Timedelta(days=2)
val_end   = FORECAST_DATE

best_params_list = []

In [0]:
retune_locations = pd.read_csv(ERRORS_PATH + "retune_history.csv",parse_dates=['DATE'])
retune_locations['DATE'] = retune_locations['DATE'] + pd.Timedelta(days=1)
LOCATIONS = retune_locations[retune_locations['DATE'] == FORECAST_DATE]['LOCATION'].unique()
print("LOCATIONS to retune:",len(LOCATIONS))

LOCATIONS to retune: 0


In [0]:
#Hyperparameter tuning
if len(LOCATIONS) == 0:
    print("No locations to retune.")
else:
    for loc in LOCATIONS:
        print(f"Tuning LOCATION {loc}")
        df_loc = df[df['LOCATION'] == loc]

        # training data before validation window
        train_df = df_loc[df_loc['DATETIME'] < val_start]
        X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]

        def objective(trial):
            # hyperparameter space
            params = {
                'objective':         'regression',
                'metric':            'mae',
                'boosting_type':     'gbdt',
                'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True),
                'num_leaves':        trial.suggest_int('num_leaves', 16, 256),
                'max_depth':         trial.suggest_int('max_depth', 3, 16),
                'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
                'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.1),
                'subsample':         trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
                'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.0),
                'n_estimators':      trial.suggest_int('n_estimators', 300, 1000),
                'random_state':      42,
                'n_jobs':            1,
                'verbosity':         -1
            }
            model = lgb.LGBMRegressor(**params)

            val_df = df_loc[
                (df_loc['DATETIME'] >= val_start) &
                (df_loc['DATETIME'] <  val_end)
            ]
            X_val, y_val = val_df[FEATURE_COLS], val_df[TARGET_COL]

            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                eval_metric='mae',
                categorical_feature=CATEGORICAL_FEATURES,
                callbacks=[
                    lgb.early_stopping(stopping_rounds=30),
                    lgb.log_evaluation(period=0)
                ]
            )
            preds = model.predict(X_val)
            return mean_absolute_error(y_val, preds)

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=TOTAL_TRIALS)

        best = study.best_params.copy()
        best.update({'LOCATION': loc, 'BEST_MAE': study.best_value})
        best_params_list.append(best)

No locations to retune.


In [0]:
#save
past_params = pd.read_csv('/dbfs/mnt/thesis/models/lgbm/basic/best_params.csv',parse_dates=['TUNING_DATE'])

if len(best_params_list) == 0:
    print("No locations to retune.")
else:
    best_params_df = pd.DataFrame(best_params_list)
    best_params_df['TUNING_DATE'] = FORECAST_DATE
    best_params_df = pd.concat([past_params,best_params_df],ignore_index=True)

    assert best_params_df['LOCATION'].nunique() == 48, "Lost Locations"

    best_params_df.to_csv(
        '/dbfs/mnt/thesis/models/lgbm/basic/' +
        "best_params.csv",
        index=False
    )

print(f"Modeling complete, with {len(LOCATIONS)} newly tuned Locations")

No locations to retune.
Modeling complete, with 0 newly tuned Locations
